## 1. Setup: environment, paths, and shared helpers

In [ ]:
import os
import shutil
import sys
from pathlib import Path

# Must be set BEFORE anything imports the retinaface verifier.
os.environ.setdefault("DEEPFACE_BACKEND_ENGINE", "pytorch")


def find_root():
    """Locate the repo root (the dir containing common/config.py).

    Tries the cwd, then the notebook's own directory, then sys.path entries,
    so the notebook works from any launch directory.
    """
    starts = [Path.cwd()]
    try:                                  # path of this .ipynb, when available
        nb_path = get_ipython().get_parent()["metadata"].get("path")
        if nb_path:
            starts.append(Path(nb_path).resolve().parent)
    except Exception:                     # noqa: BLE001
        pass
    for entry in sys.path:
        if entry:
            starts.append(Path(entry))

    seen = set()
    for start in starts:
        try:
            here = start.resolve()
        except OSError:
            continue
        for cand in [here] + list(here.parents):
            if cand in seen:
                continue
            seen.add(cand)
            if (cand / "common" / "config.py").is_file():
                return cand
    raise RuntimeError(
        "could not locate the repo root (no common/config.py found). "
        "Launch Jupyter from the repository root, or add the repo to "
        "PYTHONPATH.")


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Prefer the inline backend so figures render in JupyterLab/Notebook; fall back
# to Agg (headless) so `nbconvert --execute` still works without a display.
try:
    import matplotlib
    matplotlib.use("module://matplotlib_inline.backend_inline")
except Exception:                      # noqa: BLE001
    import matplotlib
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Video, display

RESULTS = []
_MODEL_CACHE = {}


def check(step, ok, detail=""):
    """Record and print one PASS/FAIL line."""
    status = "PASS" if ok else "FAIL"
    print("[%s] %s%s" % (status, step, (" -- " + detail) if detail else ""),
          flush=True)
    RESULTS.append((step, status, detail))
    return bool(ok)


def cache_model(key, builder):
    """Build heavy models at most once per kernel session."""
    if key not in _MODEL_CACHE:
        _MODEL_CACHE[key] = builder()
    return _MODEL_CACHE[key]


def banner(title):
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72, flush=True)


banner("CELL 1  setup + environment preflight")
print("sys.executable :", sys.executable)
print("ROOT           :", ROOT)

# Homebrew (and friends) install outside the PATH a GUI-launched Jupyter
# inherits, so probe common locations and repair PATH before anything shells
# out to ffmpeg/ffprobe. common/video_info.py calls them as bare names.
EXTRA_BIN_DIRS = ["/opt/homebrew/bin", "/usr/local/bin", "/opt/local/bin",
                  "/usr/bin", "/bin", "/usr/local/sbin", "/usr/sbin"]


def ensure_on_path(names):
    """Return {name: path}; append any missing tool's dir to PATH."""
    found, added = {}, []
    for name in names:
        path = shutil.which(name)
        if path:
            found[name] = path
            continue
        for directory in EXTRA_BIN_DIRS:
            candidate = os.path.join(directory, name)
            if os.path.isfile(candidate) and os.access(candidate, os.X_OK):
                found[name] = candidate
                if directory not in added:
                    added.append(directory)
                break
    if added:
        os.environ["PATH"] = os.pathsep.join(added + [os.environ.get("PATH", "")])
        print("added to PATH   :", added)
    return found


TOOLS = ensure_on_path(["ffmpeg", "ffprobe"])
ffmpeg, ffprobe = TOOLS.get("ffmpeg"), TOOLS.get("ffprobe")
print("ffmpeg         :", ffmpeg)
print("ffprobe        :", ffprobe)
TOOLS_OK = bool(ffmpeg and ffprobe)
check("env: ffmpeg + ffprobe resolvable", TOOLS_OK,
      "" if TOOLS_OK else "install ffmpeg, or add it to PATH")
if not TOOLS_OK:
    print("\nWARNING: ffmpeg/ffprobe are not available. Pipeline A/B cells "
          "cannot run and will report FAIL with an explanation.")

weights = {
    "insightface buffalo_l (Pipeline A)":
        (Path.home() / ".insightface" / "models" / "buffalo_l").is_dir(),
    "deepface retinaface.pth (Pipeline B)":
        (Path.home() / ".deepface" / "weights" / "retinaface.pth").is_file(),
}
for name, ok in weights.items():
    print("  weight %-40s %s" % (name, "present" if ok else "MISSING"))
check("env: model weights present", all(weights.values()))

mods = {}
for name in ("cv2", "numpy", "torch", "insightface", "retinaface",
             "fastapi", "uvicorn", "httpx"):
    try:
        __import__(name)
        mods[name] = True
    except Exception as exc:                      # noqa: BLE001
        mods[name] = False
        print("  import %-12s FAILED: %s" % (name, exc))
check("env: imports resolve", all(mods.values()), "%d modules" % len(mods))

print("\nThis notebook never edits common/config.py; PIPELINE_VERSION is "
      "read-only here.")

## 2. PARAMS — the only cell to edit

In [ ]:
# >>> EDIT HERE <<<
INPUT_VIDEO = ROOT / "downloads/face_detection_test.mp4"
OUTPUT_DIR = ROOT / "notebooks" / "out"
MAX_FRAMES = 60

# Pipeline A overrides. None => keep the DEFAULT_CONFIG value.
A_DET_THRESHOLD = None      # anonymizer.det_threshold
BOX_EXPAND = None           # anonymizer.box_expand
PIXEL_BLOCKS = None         # anonymizer.pixel_blocks
MIN_BLOCK_PX = None         # anonymizer.min_block_px

# Tracker overrides. None => keep the DEFAULT_CONFIG value.
PAD_BEFORE = None
PAD_AFTER = None
BRIDGE_GAP_FRAMES = None
INTERP_DILATE = None
BRIDGE_POLICY = None        # strict | threshold | fraction | none

# Pipeline B overrides. None => keep the DEFAULT_CONFIG value.
B_DET_THRESHOLD = None
STRIDE = None               # verifier.stride (1 = check every frame)
COVERAGE_REQUIRED = None

RUN_API = False             # cell 10: True => exercise the live uvicorn service

# ---------------------------------------------------------------------------
from common import config as config_mod
from common.hashing import sha256_file

A_OVERRIDES = {"det_threshold": A_DET_THRESHOLD, "box_expand": BOX_EXPAND,
               "pixel_blocks": PIXEL_BLOCKS, "min_block_px": MIN_BLOCK_PX}
TRACKER_OVERRIDES = {"pad_before": PAD_BEFORE, "pad_after": PAD_AFTER,
                     "bridge_gap_frames": BRIDGE_GAP_FRAMES,
                     "interp_dilate": INTERP_DILATE,
                     "bridge_policy": BRIDGE_POLICY}
B_OVERRIDES = {"det_threshold": B_DET_THRESHOLD, "stride": STRIDE,
               "coverage_required": COVERAGE_REQUIRED}

# Start from DEFAULT_CONFIG and apply ONLY the overrides above.
USER_CONFIG = {}
for section, overrides in (("anonymizer", A_OVERRIDES),
                           ("tracker", TRACKER_OVERRIDES),
                           ("verifier", B_OVERRIDES)):
    applied = {k: v for k, v in overrides.items() if v is not None}
    if applied:
        USER_CONFIG[section] = applied

# limit.max_frames is part of the hashed config on purpose: a partial run is a
# different artifact and must not share an idempotency key with a full run.
if MAX_FRAMES:
    USER_CONFIG["limit"] = {"max_frames": int(MAX_FRAMES)}

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def flatten(cfg, prefix=""):
    out = {}
    for key, val in cfg.items():
        full = "%s.%s" % (prefix, key) if prefix else key
        if isinstance(val, dict):
            out.update(flatten(val, full))
        else:
            out[full] = val
    return out


banner("CELL 2  params, config diff, idempotency key")
print("INPUT_VIDEO   :", INPUT_VIDEO)
print("OUTPUT_DIR    :", OUTPUT_DIR)
print("MAX_FRAMES    :", MAX_FRAMES)

defaults = flatten(config_mod.DEFAULT_CONFIG)
effective = flatten(config_mod.canonical_config(USER_CONFIG))
diff = {k: (defaults.get(k), effective[k]) for k in sorted(effective)
        if defaults.get(k) != effective[k]}
print("\nconfig diff vs DEFAULT_CONFIG:")
if diff:
    for key, (was, now) in diff.items():
        print("  %-26s %r -> %r" % (key, was, now))
else:
    print("  (none - running on DEFAULT_CONFIG values)")

CONTENT_SHA = sha256_file(str(INPUT_VIDEO)) if INPUT_VIDEO.is_file() else ""
IDEMPOTENCY_KEY = (config_mod.content_idempotency_key(CONTENT_SHA, USER_CONFIG)
                   if CONTENT_SHA else "")
print("\ncontent sha256  :", CONTENT_SHA or "(input missing)")
print("idempotency key :", IDEMPOTENCY_KEY or "(n/a)")
check("params: input video exists", INPUT_VIDEO.is_file(), str(INPUT_VIDEO))
check("params: idempotency key derived", bool(IDEMPOTENCY_KEY))

## 3. Input check: ffprobe metadata, real decode count, frame 0

In [ ]:
import cv2
from common.video_info import collect_video_info

banner("CELL 3  input probe")
INPUT_INFO = {}
if not INPUT_VIDEO.is_file():
    check("input: file exists", False, str(INPUT_VIDEO))
elif not TOOLS_OK:
    check("input: ffprobe available", False,
          "ffprobe not found - install ffmpeg or fix PATH (cell 1)")
else:
    INPUT_INFO = collect_video_info(str(INPUT_VIDEO), max_frames=MAX_FRAMES)
    print("resolution   : %dx%d" % (INPUT_INFO["width"], INPUT_INFO["height"]))
    print("fps          : %.3f" % INPUT_INFO["fps"])
    print("codec        : %s" % INPUT_INFO["codec"])
    print("duration     : %.2fs" % INPUT_INFO["duration_seconds"])
    print("declared     : %s" % INPUT_INFO["declared_frames"])
    print("decoded      : %s (capped at MAX_FRAMES=%s)"
          % (INPUT_INFO["decoded_frames"], MAX_FRAMES or "all"))
    print("decode errors: %s" % INPUT_INFO["decode_errors"])
    check("input: ffprobe metadata read",
          INPUT_INFO["width"] > 0 and INPUT_INFO["fps"] > 0,
          "%dx%d @ %.2f fps" % (INPUT_INFO["width"], INPUT_INFO["height"],
                                INPUT_INFO["fps"]))

    cap = cv2.VideoCapture(str(INPUT_VIDEO))
    ok, frame0 = cap.read()
    cap.release()
    if ok and frame0 is not None:
        plt.figure(figsize=(10, 5.6))
        plt.imshow(frame0[:, :, ::-1])
        plt.axis("off")
        plt.title("INPUT frame 0")
        plt.tight_layout()
        plt.show()
        check("input: frame 0 decodes", True, "shape=%s" % (frame0.shape,))
    else:
        check("input: frame 0 decodes", False)

## 4. Pipeline A: SCRFD detect -> IoU track -> pixelate -> ffmpeg render

In [ ]:
from pipeline.detect import build_anonymizer_model
from pipeline.run_pipeline_a import run_pipeline_a

banner("CELL 4  Pipeline A")

det_size = tuple(config_mod.canonical_config(USER_CONFIG)["anonymizer"]["det_size"])
detector = cache_model("scrfd-%s" % (det_size,),
                       lambda: build_anonymizer_model(det_size=det_size,
                                                      ctx_id=-1))

PIPELINE_A = run_pipeline_a(str(INPUT_VIDEO), str(OUTPUT_DIR), USER_CONFIG,
                            max_frames=MAX_FRAMES, model=detector)
RENDERED = Path(PIPELINE_A["output_path"])

print("output         :", PIPELINE_A["output_path"])
print("frames decoded : %s" % PIPELINE_A["input_info"]["decoded_frames"])
print("frames rendered: %s" % PIPELINE_A["rendered_frames"])
print("frames covered : %s (blur boxes present)" % PIPELINE_A["covered_frames"])
print("tracks         : %s" % PIPELINE_A["num_tracks"])
print("bridged gaps   : %s (%s inferred frames)"
      % (PIPELINE_A["num_bridged_gaps"], PIPELINE_A["bridged_frames"]))
print("elapsed        : %ss" % PIPELINE_A["elapsed_seconds"])

check("pipeline A: output file exists", RENDERED.is_file(), RENDERED.name)
check("pipeline A: rendered == decoded",
      PIPELINE_A["rendered_frames"] == PIPELINE_A["input_info"]["decoded_frames"],
      "%s vs %s" % (PIPELINE_A["rendered_frames"],
                    PIPELINE_A["input_info"]["decoded_frames"]))

## 5. Inline playback + original/output side-by-side

In [ ]:
banner("CELL 5  playback and side-by-side")

import cv2
import numpy as np

print("anonymised output:", RENDERED)
display(Video(str(RENDERED), embed=True))


def read_at(path, indices):
    """Sequentially decode and grab the requested frames (seeking is unreliable)."""
    want = sorted(set(int(i) for i in indices))
    out = {}
    if not want:
        return out
    cap = cv2.VideoCapture(str(path))
    idx = 0
    while idx <= want[-1]:
        ok, frame = cap.read()
        if not ok or frame is None:
            break
        if idx in want:
            out[idx] = frame.copy()
        idx += 1
    cap.release()
    return out


# Prefer a bridged (inferred) frame so the leak-prone case is always shown.
bridged = [b for b in (PIPELINE_A.get("bridges") or []) if b.get("gap_frames")]
picks = []
if bridged:
    mid = (bridged[0]["first_bridged_frame"] + bridged[0]["last_bridged_frame"]) // 2
    picks.append(mid)
    print("showing bridged frame %s (gap=%s frames)"
          % (mid, bridged[0]["gap_frames"]))
else:
    print("no bridged gap in this run - showing covered frames instead")
for track in (PIPELINE_A.get("tracks") or []):
    for candidate in ((track["start"] + track["end"]) // 2, track["start"]):
        if candidate not in picks:
            picks.append(candidate)
        if len(picks) >= 3:
            break
    if len(picks) >= 3:
        break
picks = sorted(picks)[:3]

originals = read_at(INPUT_VIDEO, picks)
outputs = read_at(RENDERED, picks)
check("side-by-side: frames readable",
      bool(originals) and bool(outputs),
      "frames=%s" % picks)

if originals and outputs:
    fig, axes = plt.subplots(len(picks), 2, figsize=(13, 3.1 * len(picks)))
    axes = np.atleast_2d(axes)
    for row, idx in enumerate(picks):
        axes[row][0].imshow(originals[idx][:, :, ::-1])
        axes[row][0].set_title("original  f%d" % idx, fontsize=9)
        axes[row][1].imshow(outputs[idx][:, :, ::-1])
        axes[row][1].set_title("anonymised  f%d" % idx, fontsize=9)
        for col in (0, 1):
            axes[row][col].axis("off")
    plt.tight_layout()
    plt.show()

## 6. Pipeline B: RetinaFace verification on the rendered file from disk

In [ ]:
from verify.retinaface_check import build_verifier_model, run_verify

banner("CELL 6  Pipeline B (independent RetinaFace check)")

import cv2
import numpy as np

CROPS_DIR = OUTPUT_DIR / "crops"
CROPS_DIR.mkdir(parents=True, exist_ok=True)
verifier = cache_model("retinaface", build_verifier_model)

# Reads the RENDERED artifact back off disk, never Pipeline A's in-memory frames.
VERIFICATION = run_verify(str(RENDERED), PIPELINE_A["tracks"], USER_CONFIG,
                          model=verifier, crops_dir=str(CROPS_DIR),
                          max_frames=MAX_FRAMES)

print("threshold      : %s" % VERIFICATION["threshold"])
print("stride         : %s" % VERIFICATION["stride"])
print("coverage       : %.4f (%s/%s planned frames)"
      % (VERIFICATION["coverage"], VERIFICATION["covered_frames"],
         VERIFICATION["planned_frames"]))
print("verifier hits  : %s" % VERIFICATION["num_flags"])
for flag in VERIFICATION["flags"]:
    print("  frame %-5d t=%-7s score=%.3f near_boundary=%s crop=%s"
          % (flag["frame_index"], flag["timestamp_seconds"], flag["score"],
             flag["near_track_boundary"],
             os.path.basename(flag["crop_path"] or "-")))

check("verify B: ran over the rendered file",
      VERIFICATION["planned_frames"] > 0,
      "%s frames planned" % VERIFICATION["planned_frames"])
check("verify B: coverage complete",
      VERIFICATION["coverage"] >= 1.0,
      "coverage=%.4f" % VERIFICATION["coverage"])

hit_crops = [f for f in VERIFICATION["flags"]
             if f.get("crop_path") and Path(f["crop_path"]).is_file()]
if hit_crops:
    show = hit_crops[:6]
    fig, axes = plt.subplots(1, len(show), figsize=(2.6 * len(show), 3.0))
    axes = np.atleast_1d(axes)
    for ax, flag in zip(axes, show):
        ax.imshow(cv2.imread(flag["crop_path"])[:, :, ::-1])
        ax.set_title("f%d  %.2f" % (flag["frame_index"], flag["score"]),
                     fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    print("showing %d of %d hit crops" % (len(show), len(hit_crops)))
else:
    print("no verifier hits -> no crops to show")

## 7. Decision: the review-by-default gate

In [ ]:
from jobs import decision as decision_mod

banner("CELL 7  decision gate")
DECISION, DECISION_REASONS = decision_mod.decide(PIPELINE_A, VERIFICATION)

print("DECISION : %s" % DECISION)
for reason in DECISION_REASONS:
    print("   - %s" % reason)
if DECISION == decision_mod.SAFE:
    print("\nSAFE was earned: all checks passed, not assumed.")
else:
    print("\nNEEDS_REVIEW is the expected default; SAFE is only earned.")

check("decision: produced a terminal verdict",
      DECISION in (decision_mod.SAFE, decision_mod.NEEDS_REVIEW,
                   decision_mod.FAILED),
      DECISION)

## 8. Gate sanity checks (pure logic, no model runs)

In [ ]:
banner("CELL 8  gate must never return SAFE for a bad input")

good_a = {
    "input_info": {"declared_frames": 100, "decoded_frames": 100,
                   "decode_errors": 0, "width": 1920, "height": 1080},
    "rendered_frames": 100, "tracks": [], "bridges": [], "covered_frames": 100,
}
good_v = {"coverage": 1.0, "num_flags": 0, "decode_errors": 0}

cases = [
    ("forced verifier hit",
     good_a, dict(good_v, num_flags=3)),
    ("frame count mismatch",
     dict(good_a, rendered_frames=99), good_v),
    ("pipeline A decode error",
     dict(good_a, input_info=dict(good_a["input_info"], decode_errors=2)),
     good_v),
]

all_blocked = True
for name, a_input, v_input in cases:
    status, reasons = decision_mod.decide(a_input, v_input)
    blocked = status != decision_mod.SAFE
    all_blocked = all_blocked and blocked
    print("[%s] %-24s -> %-13s %s"
          % ("PASS" if blocked else "FAIL", name, status, reasons))
check("gate: all 3 bad inputs blocked", all_blocked)

## 9. Full job path: real Worker + SQLite in a throwaway DB

In [ ]:
from jobs.store import JobStore
from jobs.worker import Worker

banner("CELL 9  job submission + idempotency (temp DB under notebooks/out/)")

JOB_DIR = OUTPUT_DIR / "job"
JOB_DIR.mkdir(parents=True, exist_ok=True)
JOB_DB = JOB_DIR / "e2e_jobs.sqlite3"          # NOT the repo's jobs.sqlite3
if JOB_DB.exists():
    JOB_DB.unlink()
print("job db :", JOB_DB)

store = JobStore(str(JOB_DB))
worker = Worker(store, out_dir=str(JOB_DIR),
                crops_dir=str(OUTPUT_DIR / "crops_job"),
                max_frames=MAX_FRAMES)
worker.start()          # the worker runs in its own thread

job_key = config_mod.content_idempotency_key(
    sha256_file(str(INPUT_VIDEO)), USER_CONFIG)
job_id = "job_" + job_key[:12]

submissions = []
for attempt in (1, 2):
    row, duplicate = store.create_or_get(
        job_id, "nb-e2e", CONTENT_SHA, config_mod.PIPELINE_VERSION,
        config_mod.canonical_json(config_mod.canonical_config(USER_CONFIG)),
        job_key, str(INPUT_VIDEO))
    submissions.append((row["job_id"], duplicate))
    print("submit %d -> job_id=%s duplicate=%s status=%s"
          % (attempt, row["job_id"], duplicate, row["status"]))
    if not duplicate:
        worker.submit(row["job_id"])

worker.wait_until_done(job_id, timeout=1800, poll=1.0)
FINAL = store.get(job_id)

print("\nfinal status    :", FINAL["status"])
print("final decision  :", FINAL["decision"])
print("decision reasons:", FINAL["decision_reasons_json"])
print("output path     :", FINAL["output_path"])
print("attempt_count   :", FINAL["attempt_count"])
print("frame counts    :", FINAL["frame_counts_json"])

check("job: both submissions -> one job_id",
      submissions[0][0] == submissions[1][0], submissions[0][0])
check("job: second submission reported duplicate",
      submissions[1][1] is True)
check("job: exactly one row in the store", store.count_jobs() == 1,
      "rows=%d" % store.count_jobs())
check("job: processed exactly once",
      (FINAL["attempt_count"] or 0) == 1,
      "attempt_count=%s" % FINAL["attempt_count"])
check("job: reached a terminal decision",
      FINAL["status"] in ("SAFE", "NEEDS_REVIEW", "FAILED"), FINAL["status"])
check("job: temp DB under notebooks/out, repo jobs.sqlite3 untouched",
      JOB_DB.exists() and JOB_DB.parent == JOB_DIR, JOB_DB.name)

## 10. Optional: live HTTP service (guarded by RUN_API)

In [ ]:
banner("CELL 10  live uvicorn API")

if not RUN_API:
    print("SKIPPED - set RUN_API = True in the PARAMS cell to run this step.")
    check("api: optional live service", True, "skipped (RUN_API=False)")
else:
    import socket
    import subprocess
    import time

    import httpx

    probe = socket.socket()
    probe.bind(("127.0.0.1", 0))
    port = probe.getsockname()[1]
    probe.close()
    print("using free port", port)

    env = dict(os.environ)
    env["ANON_WORKER"] = "1"                     # in-process worker
    env["PYTHONPATH"] = str(ROOT)
    env["DEEPFACE_BACKEND_ENGINE"] = "pytorch"
    env["JOBS_DB"] = str(OUTPUT_DIR / "api" / "api_jobs.sqlite3")
    env["ARTIFACT_DIR"] = str(OUTPUT_DIR / "api")
    env["CROPS_DIR"] = str(OUTPUT_DIR / "api" / "crops")
    Path(env["JOBS_DB"]).parent.mkdir(parents=True, exist_ok=True)

    server = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "jobs.api:app",
         "--port", str(port), "--log-level", "warning"],
        cwd=str(ROOT), env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    base = "http://127.0.0.1:%d" % port
    try:
        health = None
        for _ in range(60):
            try:
                health = httpx.get(base + "/health", timeout=2.0)
                break
            except Exception:                     # noqa: BLE001
                time.sleep(1.0)
        print("health:", health.text if health is not None else "no response")

        body = {"upload_id": "nb-api", "video_path": str(INPUT_VIDEO),
                "config": USER_CONFIG}
        first = httpx.post(base + "/jobs", json=body, timeout=60.0).json()
        second = httpx.post(base + "/jobs", json=body, timeout=60.0).json()
        print("POST 1:", first)
        print("POST 2:", second)
        got = httpx.get(base + "/jobs/" + first["job_id"],
                        timeout=60.0).json()
        print("GET:", got["job_id"], got["status"], got["decision"])

        check("api: same job_id from two POSTs",
              first["job_id"] == second["job_id"], first["job_id"])
        check("api: second POST reported duplicate",
              second.get("duplicate") is True)
        check("api: GET returns the same job",
              got["job_id"] == first["job_id"])
    finally:
        server.terminate()
        try:
            server.wait(timeout=15)
        except Exception:                         # noqa: BLE001
            server.kill()

## 11. Summary

In [ ]:
banner("CELL 11  summary")

width = max(len(step) for step, _, _ in RESULTS)
print("%-*s  %-6s %s" % (width, "STEP", "RESULT", "DETAIL"))
print("-" * (width + 40))
for step, status, detail in RESULTS:
    print("%-*s  %-6s %s" % (width, step, status, detail))

passed = sum(1 for _, status, _ in RESULTS if status == "PASS")
failed = [step for step, status, _ in RESULTS if status == "FAIL"]
print("\n%d/%d checks PASS" % (passed, len(RESULTS)))
if failed:
    print("FAILED: " + ", ".join(failed))
else:
    print("no failures")

print("\nOutput files (all under %s):" % OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print("  %-58s %8.1f KB"
              % (path.relative_to(ROOT), path.stat().st_size / 1024.0))